In [ ]:
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getMostRecentRowWithDataBefore
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId

In [ ]:
client = makeEfdClient()
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=True)

In [ ]:
from lsst.ts.ofc.state_estimator import StateEstimator
from lsst.ts.ofc import OFCData
ofcData = OFCData()
ofcData.configure_controller()
await ofcData.configure_instrument('lsst')

# Restrict to standard_22 DOFs (5+5+7+5 = 22)
m2_hexapod = np.ones(5, dtype=bool)
cam_hexapod = np.ones(5, dtype=bool)
m1m3_bending = np.zeros(20, dtype=bool)
m2_bending = np.zeros(20, dtype=bool)
m1m3_bending[:7] = True
m2_bending[:5] = True
ofcData.comp_dof_idx = dict(
    m2HexPos=m2_hexapod,
    camHexPos=cam_hexapod,
    M1M3Bend=m1m3_bending,
    M2Bend=m2_bending,
)
state_estimator = StateEstimator(ofcData)


In [ ]:
ofcData.comp_dof_idx

In [ ]:
day_obs = 20260404
firstSeqNum = 14
lastSeqNum = 646
firstExpId = int(day_obs*1e5 + firstSeqNum)
lastExpId = int(day_obs*1e5 + lastSeqNum)
dataId = {'exposure': firstExpId, 'instrument':'LSSTCam'}
firstExpRecord = getExpRecordFromDataId(butler, dataId)
dataId = {'exposure': lastExpId, 'instrument':'LSSTCam'}
lastExpRecord = getExpRecordFromDataId(butler, dataId)

In [ ]:
wavefront_errors =getEfdData(client, 
    topic="lsst.sal.MTAOS.logevent_wavefrontError",
    columns=[
        f"nollZernikeValues{i}" for i in range(27)
    ] + [
        f"nollZernikeIndices{i}" for i in range(27)
    ] + [
        "sensorId",
        "visitId"
    ],
    begin=firstExpRecord.timespan.begin,
    end=lastExpRecord.timespan.end                         
)
print(len(wavefront_errors))

dof_event = getEfdData(
    client,
    topic="lsst.sal.MTAOS.logevent_degreeOfFreedom",
    begin=firstExpRecord.timespan.begin,
    end=lastExpRecord.timespan.end                         
)
print(len(dof_event))

In [ ]:
expId = 2026040400398
this_dof_event = dof_event[dof_event['visitId']==expId]
this_dof_event.columns
i=12
this_dof_event[f'visitDoF{i}']

In [ ]:
dof_correct

In [ ]:
indices = list(range(0, 17)) + list(range(30,35))
dof_correct = []
for i in range(50):
    dof = this_dof_event[f'visitDoF{i}'].values[0]
    dof_correct.append(dof)
dof_correct = np.array(dof_correct)[indices]
dof_vector = np.zeros([50])
dof_vector[ofcData.dof_idx] = dof_correct
vmodes = state_estimator.get_vmodes_from_dofs(dof_vector)[0:12]
vmodes

In [ ]:
exposureList = []
for record in butler.registry.queryDimensionRecords("exposure", 
                    where=f"exposure.day_obs={day_obs} and instrument='LSSTCam'"):
    exposureList.append([record.id, record])
exposureList.sort(key=lambda x: x[0])
z4s = []
z11s = []
m2dzs = []
camdzs = []
vmode5s = []
vmode10s = []
for [expId, record] in exposureList:
    seqNum = int(expId - 1e5*day_obs)
    try:
        these_wavefront_errors = wavefront_errors[wavefront_errors['visitId']==expId]
        this_dof_event = dof_event[dof_event['visitId']==expId]
        z4 = these_wavefront_errors['nollZernikeValues0'].values.mean()
        z11 = these_wavefront_errors['nollZernikeValues7'].values.mean()
        m2dz = this_dof_event['visitDoF0'].values[0]
        camdz = this_dof_event['visitDoF5'].values[0]
        dof_correct = []
        for i in range(50):
            dof = this_dof_event[f'visitDoF{i}'].values[0]
            dof_correct.append(dof)
        dof_correct = np.array(dof_correct)[indices]
        dof_vector = np.zeros([50])
        dof_vector[ofcData.dof_idx] = dof_correct
        vmodes = state_estimator.get_vmodes_from_dofs(dof_vector)[0:12]
        z4s.append(z4)
        z11s.append(z11)
        m2dzs.append(m2dz)
        camdzs.append(camdz)
        vmode5s.append(vmodes[4])
        vmode10s.append(vmodes[9])
    except:
        continue
print(len(z4s), len(z11s), len(m2dzs), len(camdzs), len(vmode5s), len(vmode10s))
        


In [ ]:
fig, axs = plt.subplots(2,4,figsize=(20,5))
axes = []
for i in range(2):
    for j in range(4):
        axes.append(axs[i][j])
fig.suptitle(f"Z4 correlations {day_obs}")
plt.subplots_adjust(wspace=0.5, hspace=0.5)
axes[0].scatter(m2dzs, z4s)
axes[0].set_title("M2 dz")
axes[0].set_ylabel("Z4 (microns)")
axes[0].set_xlabel("M2 dz (microns)")
axes[0].set_ylim(-1.5,1)
axes[0].set_xlim(-75,75)
axes[1].scatter(camdzs, z4s)
axes[1].set_title("Cam dz")
axes[1].set_ylabel("Z4 (microns)")
axes[1].set_xlabel("Cam dz (microns)")
axes[1].set_ylim(-1.5,1)
axes[1].set_xlim(-75,75)
vm = 0.6 * np.array(m2dzs) + 0.4 * np.array(camdzs)
axes[2].scatter(vm, z4s)
axes[2].set_title("0.6*m2dz + 0.4*camdz")
axes[2].set_ylabel("Z4 (microns)")
axes[2].set_xlabel("0.6*m2dz + 0.4*camdz (microns)")
axes[2].set_ylim(-1.5,1)
axes[2].set_xlim(-10,10)
axes[3].scatter(vmode5s, z4s)
axes[3].set_title("Vmode5")
axes[3].set_ylabel("Z4 (microns)")
axes[3].set_xlabel("Vmode5 ")
axes[3].set_ylim(-1.5,1)
axes[3].set_xlim(-0.5,0.5)
vm2 = 0.6 * np.array(m2dzs) - 0.4 * np.array(camdzs)
axes[4].scatter(vm2, z11s)
axes[4].set_title("0.6*m2dz - 0.4*camdz (microns)")
axes[4].set_ylabel("Z11 (microns)")
axes[4].set_xlabel("0.6*m2dz - 0.4*camdz (microns)")
#axes[4].set_ylim(-1.5,1)
#axes[4].set_xlim(-0.5,0.5)
axes[5].scatter(vmode10s, z11s)
axes[5].set_title("Vmode10")
axes[5].set_ylabel("Z11 (microns)")
axes[5].set_xlabel("Vmode10 ")
#axes[5].set_ylim(-1.5,1)
#axes[5].set_xlim(-0.5,0.5)

fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Z4_Correlations_{day_obs}.png")